In [5]:
# Basic MLP for tabular data using scikit-learn
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report

In [6]:
df = pd.read_csv("health care diabetes.csv")  # change path
X = df.drop(columns=["Outcome"])
y = df["Outcome"]

In [7]:
# Split & scale
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [8]:
mlp = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    solver="adam",
    max_iter=200,
    random_state=42
)

In [9]:
mlp.fit(X_train, y_train)

c:\Users\zheng\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


MLPClassifier(hidden_layer_sizes=(64, 32), random_state=42)

In [10]:
# Eva
y_pred = mlp.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.80      0.78      0.79       100
           1       0.61      0.63      0.62        54

    accuracy                           0.73       154
   macro avg       0.70      0.70      0.70       154
weighted avg       0.73      0.73      0.73       154



In [11]:
mlp.hidden_layer_sizes

(64, 32)

In [13]:
import numpy as np

# After you've trained `mlp` and fit `scaler` (from the earlier code):
# mlp.hidden_layer_sizes should match what you used, e.g. (64, 32)

def relu(x): 
    return np.maximum(0, x)

def hidden_embeddings(X_raw, scaler, mlp, layer_index=-1):
    """
    Returns activations from the chosen hidden layer (default: last hidden layer).
    X_raw: unscaled features
    layer_index: 0 for first hidden layer, 1 for second, etc. -1 means last
    """
    X = scaler.transform(X_raw).astype(np.float64)  # sklearn stores weights as float64
    activ = X
    n_hidden = len(mlp.coefs_) - 1  # last weight matrix is to output layer

    # resolve negative index
    if layer_index < 0:
        layer_index = n_hidden + layer_index

    for i in range(n_hidden):
        W, b = mlp.coefs_[i], mlp.intercepts_[i]
        activ = relu(activ @ W + b)     # sklearn MLPClassifier default: ReLU if activation="relu"
        if i == layer_index:            # stop at the desired hidden layer
            return activ
    return activ

# Example usage:
# X_train_raw, X_test_raw are your original (unscaled) splits
emb_train = hidden_embeddings(X_train, scaler, mlp, layer_index=-1)  # shape: (n_train, 32)
emb_test  = hidden_embeddings(X_test,  scaler, mlp, layer_index=-1)  # shape: (n_test, 32)

c:\Users\zheng\anaconda3\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\zheng\anaconda3\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [14]:
print(emb_train.shape, emb_test.shape)  # (n_samples, size_of_last_hidden_layer)

(614, 32) (154, 32)


In [21]:
emb_train_df = pd.DataFrame(emb_train)
emb_train_df.head()

,0,1,2,3,4,5,6,7,8,9,...,22,23,24,25,26,27,28,29,30,31
0,0.251114,0.236072,0.0,0.0,4.063727,1.491825,3.131240,1.207131,0.723453,3.255871,...,1.517910,0.0,4.934466,0.0,5.180281,0.0,5.240861,2.750411,1.312699,1.601056
1,0.295112,0.000000,0.0,0.0,4.112956,1.714657,3.416641,0.655326,0.733822,3.692367,...,2.057878,0.0,5.159543,0.0,4.742851,0.0,5.035167,2.370888,1.446814,1.805440
2,0.071890,0.000000,0.0,0.0,4.465738,1.853892,4.287295,0.000000,1.047151,4.565599,...,2.659292,0.0,5.639159,0.0,4.338850,0.0,5.110163,1.903395,2.013104,2.592455
3,0.255161,0.198464,0.0,0.0,4.069105,1.493535,3.156603,1.092523,0.717634,3.286887,...,1.599145,0.0,4.944385,0.0,5.124175,0.0,5.193350,2.726317,1.338901,1.633260
4,0.114933,0.000000,0.0,0.0,4.350402,1.756138,3.797482,0.059765,0.821782,4.130259,...,2.200346,0.0,5.361036,0.0,4.806594,0.0,5.175996,2.384518,1.658687,2.175711
